# 01 — EDA mapa de rendimiento

Exploración del monitor de cosecha de maíz (**Oviedo - Lote1**).

La primera lectura del Excel puede tardar varios minutos. Después se usa el Parquet en `data/processed/`.

**Colab:** File → Open notebook → GitHub → `FFerrerPuccio1/Ciencia-Datos-G4`. La celda de setup clona el repo (código + Excel) porque Colab solo abre el `.ipynb`. Si el repo es privado, cargá un secreto `GITHUB_TOKEN` en Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_SLUG = "FFerrerPuccio1/Ciencia-Datos-G4"


def _in_colab() -> bool:
    return "google.colab" in sys.modules


def _github_token() -> str:
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token or not _in_colab():
        return token
    try:
        from google.colab import userdata

        return (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        return ""


def _clone_url() -> str:
    token = _github_token()
    if token:
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    return f"https://github.com/{REPO_SLUG}.git"


def project_root() -> Path:
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "src" / "cdd").is_dir():
            return folder

    if not _in_colab():
        raise FileNotFoundError(
            "No se encontró src/cdd. Ejecutá el notebook desde el repo "
            "(carpeta notebooks/ o raíz del proyecto)."
        )

    dest = Path("/content") / "Ciencia-Datos-G4"
    if not (dest / "src" / "cdd").is_dir():
        print(f"Clonando https://github.com/{REPO_SLUG} ...")
        subprocess.check_call(["git", "clone", "--depth", "1", _clone_url(), str(dest)])
    os.chdir(dest)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl", "pyarrow"]
    )
    return dest


ROOT = project_root()
SRC = str(ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from cdd.cleaning import clean_yield_map
from cdd.io import load_processed, load_raw, to_parquet
from cdd.paths import PROCESSED_PARQUET

print("Proyecto:", ROOT)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 1. Carga Excel → Parquet

In [ ]:
if PROCESSED_PARQUET.exists():
    df = load_processed()
    print(f"Leído Parquet: {PROCESSED_PARQUET}")
else:
    print("Leyendo Excel (puede tardar)...")
    df = load_raw()
    to_parquet(df)
    print(f"Guardado Parquet: {PROCESSED_PARQUET}")

print(df.shape)
df.head()

## 2. Esquema, nulos y rangos

In [ ]:
df.info()
display(df.describe(include="all").T)
nulos = df.isna().sum().rename("nulos")
nulos_pct = (df.isna().mean() * 100).rename("pct")
display(pd.concat([nulos, nulos_pct], axis=1))

## 3. Limpieza mínima

Tipos numéricos, `Time` de serial Excel a datetime, y filtros de coordenadas nulas, velocidad 0.1–20 km/h, rendimiento ≥ 0 y humedad 0–40 %.

In [ ]:
clean = clean_yield_map(df)
print(f"Filas crudas: {len(df):,}")
print(f"Filas limpias: {len(clean):,}")
print(f"Descartadas: {len(df) - len(clean):,}")
clean[["Time", "Yld_Mass_D", "Moisture__", "Speed_km_h", "lat", "lon"]].head()

## 4. Distribuciones de rendimiento, humedad y velocidad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
vars_ = [("Yld_Mass_D", "Rendimiento seco"), ("Moisture__", "Humedad %"), ("Speed_km_h", "Velocidad km/h")]
for ax, (col, title) in zip(axes, vars_):
    sns.histplot(clean[col], bins=50, ax=ax, kde=True)
    ax.set_title(title)
fig.tight_layout()
plt.show()

clean[list(v[0] for v in vars_)].describe()

## 5. Mapa de puntos por `Yld_Mass_D`

In [ ]:
sample = clean.sample(n=min(40_000, len(clean)), random_state=42)

fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(
    sample["lon"],
    sample["lat"],
    c=sample["Yld_Mass_D"],
    s=1,
    cmap="YlGn",
    alpha=0.7,
)
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.set_title("Mapa de puntos — rendimiento seco (muestra)")
ax.set_aspect("equal", adjustable="box")
fig.colorbar(sc, ax=ax, label="Yld_Mass_D")
plt.show()

## 6. Comparación por pasada (`Dataset`)

In [ ]:
por_pasada = (
    clean.groupby("Dataset", observed=True)
    .agg(
        n=("fid", "size"),
        yld_d_mediana=("Yld_Mass_D", "median"),
        humedad_mediana=("Moisture__", "median"),
        vel_mediana=("Speed_km_h", "median"),
    )
    .sort_values("n", ascending=False)
)
display(por_pasada)

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=clean, x="Dataset", y="Yld_Mass_D", ax=ax, showfliers=False)
ax.tick_params(axis="x", rotation=45)
ax.set_title("Rendimiento seco por Dataset / pasada")
fig.tight_layout()
plt.show()

## 7. Notas de calidad de dato

Outliers típicos de un monitor de cosecha:

- **Arranques y cabeceras**: velocidad baja o flujo inestable al entrar/salir de la pasada.
- **Solapes**: el ancho de labor (`Swth_Wdth_`) fijo no siempre refleja el solape real.
- **Humedad y masa**: picos de `Yld_Mass_W` / `Yld_Mass_D` en frenadas o cambios de rumbo.
- **Tiempo**: el serial de Excel se convierte a datetime; revisar zona horaria si se cruza con otras fuentes.
- **GPS**: puntos con `lat`/`lon` nulos o fuera del lote se filtran en la limpieza mínima; aún pueden quedar puntos de borde.

Siguientes pasos fuera de este notebook: filtrado de bordes, interpolación espacial y zonificación.